#### Ingesting FHVHV Trips Data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from dotenv import load_dotenv
import logging
import os

load_dotenv()

##### Creating directory for fhvhv log file 

In [ ]:
log_file = r"/app/data/logs/"
os.makedirs(name = log_file,exist_ok= True)
file_name_fhvhv = os.path.join(log_file,'fhvhv_trips.log')
print(file_name_fhvhv)

##### Adding logger

In [ ]:
logger = logging.getLogger(__name__)
logger.propagate = False   # <-- add this
logger.setLevel(logging.DEBUG)
fh = logging.FileHandler(file_name_fhvhv,mode = 'w')
logger.addHandler(fh)
formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')
fh.setFormatter(formatter)

##### Creating spark session

In [4]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("NYC Taxi Pipeline")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.5.0,"
        "software.amazon.awssdk:bundle:2.31.54"
    )
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.endpoint", os.environ.get("S3_ENDPOINT", "s3.amazonaws.com"))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions",16)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

##### Schema for HVFHV Data

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, TimestampType
)

fhvhv_schema = StructType([
    StructField('hvfhs_license_num',     StringType(),       True),
    StructField('dispatching_base_num',  StringType(),       True),
    StructField('originating_base_num',  StringType(),       True),
    StructField('request_datetime',      TimestampType(), True),
    StructField('on_scene_datetime',     TimestampType(), True),
    StructField('pickup_datetime',       TimestampType(), True),
    StructField('dropoff_datetime',      TimestampType(), True),
    StructField('PULocationID',          IntegerType(),      True),
    StructField('DOLocationID',          IntegerType(),      True),
    StructField('trip_miles',            DoubleType(),       True),
    StructField('trip_time',             LongType(),         True),
    StructField('base_passenger_fare',   DoubleType(),       True),
    StructField('tolls',                 DoubleType(),       True),
    StructField('bcf',                   DoubleType(),       True),
    StructField('sales_tax',             DoubleType(),       True),
    StructField('congestion_surcharge',  DoubleType(),       True),
    StructField('airport_fee',           DoubleType(),       True),
    StructField('tips',                  DoubleType(),       True),
    StructField('driver_pay',            DoubleType(),       True),
    StructField('shared_request_flag',   StringType(),       True),
    StructField('shared_match_flag',     StringType(),       True),
    StructField('access_a_ride_flag',    StringType(),       True),
    StructField('wav_request_flag',      StringType(),       True),
    StructField('wav_match_flag',        StringType(),       True),
    StructField('cbd_congestion_fee',    DoubleType(),       True),
])

##### Reading HVFHV file

In [6]:
fhvhv_data = spark.read\
                .option('header',True)\
                    .schema(fhvhv_schema)\
                        .parquet('/app/data/input/hvfhv/fhvhv_tripdata_2026-04.parquet')

In [7]:
fhvhv_data.printSchema()

In [8]:
fhvhv_data.rdd.getNumPartitions()

##### Log Raw Count

In [9]:
logger.info(f'Raw count: {fhvhv_data.count()}')

##### Cleansing data outside of April 2026 Window

In [10]:
fhvhv_data = fhvhv_data.filter(~(date_format(col('pickup_datetime'),'yyyy-MM') >= '2026-05'))

In [11]:
fhvhv_data = fhvhv_data.filter(~(date_format(col('pickup_datetime'),'yyyy-MM') < '2026-04'))

##### Removing data if location data is missing

In [12]:
fhvhv_data = fhvhv_data.filter(~(col('PUlocationID').isNull() & col('DOlocationID').isNull()))

##### Checking if pickup and drop datetime are identical (data quality issue)

In [13]:
fhvhv_DQ_cnt = fhvhv_data.filter(col('pickup_datetime') >= col('dropOff_datetime')).count()

In [14]:
if fhvhv_DQ_cnt != 0:
    fhvhv_data = fhvhv_data.filter(~(col('pickup_datetime') == col('dropOff_datetime')))


##### Log count after validation

In [15]:
logger.info(f"After Validation Count: {fhvhv_data.count()}")

##### Derived Column trip duration in minutes

In [16]:
from pyspark.sql.functions import timestamp_diff
fhvhv_data = fhvhv_data.withColumn('trip_duration_minutes',timestamp_diff('minute',col('pickup_datetime'),col('dropOff_datetime')))

##### Clearing any trips with zero minutes trip duration

In [17]:
fhvhv_data = fhvhv_data.filter(~(col('trip_duration_minutes') == 0))

##### Adding source filename and ingestion timestamp columns

In [18]:
fhvhv_data = fhvhv_data.withColumns({'source_file': lit('fhvhv_tripdata_2026-04.parquet'),
'ingestion_timestamp' : current_timestamp()})

##### Writing data with paritioning in S3 Bucket

In [19]:
s3_bucket = os.environ["S3_BUCKET"]

fhvhv_data \
    .withColumn('pickup_date', date_format(col('pickup_datetime'), 'yyyy-MM-dd')) \
    .write \
    .option('header', True) \
    .partitionBy('pickup_date') \
    .mode('overwrite') \
    .parquet(f's3a://{s3_bucket}/fhvhv')

##### Data Count written to S3

In [20]:
logger.info(f'Complete data count written to S3: {fhvhv_data.count()}')